In [1]:
#Google Maps Places API scraper for LPG infrastructure. Searches for primary depots and filling stations separately, and saves the results as two layers in a GeoPackage (EPSG:3857).

In [ ]:
# Cell 1: Import libraries and set up API key
import requests
import time
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from collections import Counter

DATA_DIR = "dataset_big"
OUTPUT_GPKG = f"{DATA_DIR}/lpg_infrastructure_3857.gpkg"

# ==================== CONFIGURATION ====================
API_KEY = "key"          # Replace with your actual API key

# Nigeria bounding box (change for other countries)
COUNTRY_BBOX = {
    'min_lat': 4.0,
    'max_lat': 14.0,
    'min_lon': 2.5,
    'max_lon': 14.5
}

STEP = 0.65            # degrees (~72 km) – ensures overlap with 50 km radius
RADIUS = 50000         # search radius in meters

# Keywords for primary depots (industrial storage, terminals)
DEPOT_KEYWORDS = [
    "LPG depot",
    "LPG terminal",
    "LPG storage",
    "LPG primary depot"
]

# Keywords for filling stations (retail, cylinder refill points)
FILLING_KEYWORDS = [
    "LPG cylinder filling plant",
    "LPG bottling plant",
    "gas bottling plant",
    "LPG filling plant",
    "cylinder filling station",
    "LPG bottling station"
]


In [3]:
# Cell 2: Generate grid of search centers
def generate_grid(bbox, step):
    """Generate a grid of search center points covering the bounding box."""
    points = []
    lat = bbox['min_lat']
    while lat <= bbox['max_lat']:
        lon = bbox['min_lon']
        while lon <= bbox['max_lon']:
            points.append((round(lat, 4), round(lon, 4)))
            lon += step
        lat += step
    return points

In [4]:
# Cell 3: Nearby search function with pagination
def nearby_search(lat, lon, keyword, radius=50000):
   
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': radius,
        'keyword': keyword,
        'key': API_KEY
    }

    while True:
        resp = requests.get(url, params=params)
        data = resp.json()

        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            print(f"API error: {data['status']} - {data.get('error_message', '')}")
            break

        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': keyword,
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0)
            })

        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)      # token needs a short delay to become valid
        else:
            break

    return results

In [ ]:
# Cell 4: preparation of the search and saving results
def search_all(keywords, grid, label):
    """
    Run a search for a list of keywords over the entire grid.
    Returns a list of unique places (dictionaries) found.
    """
    all_places = {}          # dict keyed by place_id to avoid duplicates inside this category
    total_api_calls = 0
    error_count = 0
    first_error = None

    for idx, (lat, lon) in enumerate(grid):
        if (idx + 1) % 10 == 0 or idx == 0:
            print(f"Processing grid point {idx+1}/{len(grid)} for {label}: ({lat}, {lon})")

        for kw in keywords:
            try:
                places = nearby_search(lat, lon, kw)
                # Estimate number of API calls (first page + each additional page)
                total_api_calls += 1 + (len(places) // 60)
                for p in places:
                    if p['place_id'] not in all_places:
                        all_places[p['place_id']] = p
            except Exception as e:
                error_count += 1
                if first_error is None:
                    first_error = f"{kw} at ({lat},{lon}): {e}"
                    print(f"  ⚠️ First error: {first_error}")

            time.sleep(0.2)    # polite delay to avoid hitting rate limits

        # Save a checkpoint every 10 grid points
        if (idx + 1) % 10 == 0:
            checkpoint_file = f"{DATA_DIR}/{label}_interim.json"
            with open(checkpoint_file, 'w') as f:
                json.dump(list(all_places.values()), f, indent=2)
            print(f"  Checkpoint: {len(all_places)} unique places, {error_count} errors")

    print(f"\nFinished {label}. Estimated API calls: {total_api_calls}")
    print(f"Unique places found: {len(all_places)}")
    if error_count > 0:
        print(f"Total errors: {error_count}, first error: {first_error}")

    return list(all_places.values())

def save_to_gpkg(results, layer_name, gpkg_path):
    """Save a list of places as a GeoPackage layer in EPSG:3857."""
    if not results:
        print(f"No results for layer '{layer_name}', skipping.")
        return

    df = pd.DataFrame(results)
    geometry = [Point(lon, lat) for lon, lat in zip(df['lng'], df['lat'])]
    gdf = gpd.GeoDataFrame(
        df[['place_id', 'name', 'address', 'types', 'keyword', 'search_lat', 'search_lon', 'rating', 'user_ratings_total']],
        geometry=geometry,
        crs="EPSG:4326"
    )
    # Keep original coordinates as attributes
    gdf['lat'] = df['lat']
    gdf['lon'] = df['lng']

    # Reproject to Web Mercator (EPSG:3857)
    gdf_3857 = gdf.to_crs("EPSG:3857")
    gdf_3857.to_file(gpkg_path, driver='GPKG', layer=layer_name)
    print(f"  ✅ Layer '{layer_name}' saved to {gpkg_path} with {len(gdf_3857)} points.")


In [ ]:
# Cell 6: main run
if __name__ == "__main__":
    print("Generating search grid...")
    grid = generate_grid(COUNTRY_BBOX, STEP)
    print(f"Grid has {len(grid)} points.\n")

    # Search for primary depots
    print("=== SEARCHING FOR PRIMARY DEPOTS ===\n")
    depot_places = search_all(DEPOT_KEYWORDS, grid, "depot")

    # Search for primary filling plants
    print("\n=== SEARCHING FOR PRIMARY FILLING PLANTS ===\n")
    filling_places = search_all(FILLING_KEYWORDS, grid, "filling")

    # Save both layers to the same GeoPackage
    print("\nSaving results to GeoPackage...")
    save_to_gpkg(depot_places, "depots", OUTPUT_GPKG)
    save_to_gpkg(filling_places, "filling_plants", OUTPUT_GPKG)
    print(f"\nDone. All data saved in '{OUTPUT_GPKG}'.")

Generating search grid...
Grid has 304 points.

=== SEARCHING FOR PRIMARY DEPOTS ===

Processing grid point 1/304 for depot: (4.0, 2.5)
Processing grid point 10/304 for depot: (4.0, 8.35)
  Checkpoint: 0 unique places, 0 errors
Processing grid point 20/304 for depot: (4.65, 2.5)
  Checkpoint: 1 unique places, 0 errors
Processing grid point 30/304 for depot: (4.65, 9.0)
  Checkpoint: 20 unique places, 0 errors
Processing grid point 40/304 for depot: (5.3, 3.15)
  Checkpoint: 20 unique places, 0 errors
Processing grid point 50/304 for depot: (5.3, 9.65)
  Checkpoint: 28 unique places, 0 errors
Processing grid point 60/304 for depot: (5.95, 3.8)
  Checkpoint: 28 unique places, 0 errors
Processing grid point 70/304 for depot: (5.95, 10.3)
  Checkpoint: 45 unique places, 0 errors
Processing grid point 80/304 for depot: (6.6, 4.45)
  Checkpoint: 80 unique places, 0 errors
Processing grid point 90/304 for depot: (6.6, 10.95)
  Checkpoint: 87 unique places, 0 errors
Processing grid point 100/3

In [ ]:
# Cell: Load results from checkpoint files and print statistics
import json
import os
from collections import Counter

print("\n=== FINAL STATISTICS ===\n")

# Load depots
depot_places = []
if os.path.exists(f'{DATA_DIR}/depot_interim.json'):
    with open(f'{DATA_DIR}/depot_interim.json', 'r') as f:
        depot_places = json.load(f)
    print(f"Depots: loaded {len(depot_places)} records")
else:
    print("Depot file not found")

# Load filling plants
filling_places = []
if os.path.exists(f'{DATA_DIR}/filling_interim.json'):
    with open(f'{DATA_DIR}/filling_interim.json', 'r') as f:
        filling_places = json.load(f)
    print(f"Filling plants: loaded {len(filling_places)} records")
else:
    print("Filling plant file not found")

print("\n" + "="*50)

# Depot statistics
print("\n--- DEPOT STATISTICS ---")
print(f"Total depots: {len(depot_places)}")
if depot_places:
    kw_depot = Counter([p['keyword'] for p in depot_places])
    print("By keyword:")
    for kw, cnt in kw_depot.most_common():
        print(f"  {kw}: {cnt}")
    
    with_reviews = sum(1 for p in depot_places if p.get('user_ratings_total', 0) > 0)
    pct = (with_reviews / len(depot_places)) * 100
    print(f"Places with reviews: {with_reviews} ({pct:.1f}%)")
    
    print("First 5 depots:")
    for p in depot_places[:5]:
        print(f"  - {p.get('name', 'N/A')}")

# Filling plant statistics
print("\n--- FILLING PLANT STATISTICS ---")
print(f"Total filling plants: {len(filling_places)}")
if filling_places:
    kw_fill = Counter([p['keyword'] for p in filling_places])
    print("By keyword:")
    for kw, cnt in kw_fill.most_common():
        print(f"  {kw}: {cnt}")
    
    with_reviews = sum(1 for p in filling_places if p.get('user_ratings_total', 0) > 0)
    pct = (with_reviews / len(filling_places)) * 100
    print(f"Places with reviews: {with_reviews} ({pct:.1f}%)")
    
    print("First 5 filling plants:")
    for p in filling_places[:5]:
        print(f"  - {p.get('name', 'N/A')}")

print("\nDone.")


=== FINAL STATISTICS ===

Depots: loaded 171 records
Filling plants: loaded 605 records


--- DEPOT STATISTICS ---
Total depots: 171
By keyword:
  LPG storage: 138
  LPG depot: 18
  LPG primary depot: 10
  LPG terminal: 5
Places with reviews: 138 (80.7%)
First 5 depots:
  - YAOUNDE MAGZI TOTAL
  - Nigeria Liquefied Natural Gas (LNG), Gbaran Ubie
  - Hi5 Cooking (LPG) Gas Supplies
  - Falcon Corporation Limited 15000MT LPG Depot
  - Falcon corporation Ltd - LPG Depot

--- FILLING PLANT STATISTICS ---
Total filling plants: 605
By keyword:
  cylinder filling station: 270
  LPG filling plant: 146
  LPG bottling station: 93
  LPG cylinder filling plant: 93
  LPG bottling plant: 2
  gas bottling plant: 1
Places with reviews: 557 (92.1%)
First 5 filling plants:
  - Vala Enterprise
  - AFRICA CYLINDER COMPANY
  - total station Odza terminal 10
  - YAOUNDE MAGZI TOTAL
  - Nigeria Liquefied Natural Gas (LNG), Gbaran Ubie

Done.


In [ ]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# Load data from JSON files (if you have them, otherwise read from GPKG)
with open(f'{DATA_DIR}/depot_interim.json') as f:
    depot_data = json.load(f)
with open(f'{DATA_DIR}/filling_interim.json') as f:
    filling_data = json.load(f)

def create_gdf(data, cat):
    df = pd.DataFrame(data)
    if df.empty:
        return gpd.GeoDataFrame(columns=['place_id','name','keyword','lat','lng'], geometry=[], crs='EPSG:4326')
    geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
    gdf = gpd.GeoDataFrame(df[['place_id','name','keyword','lat','lng']], geometry=geom, crs='EPSG:4326')
    gdf['category'] = cat
    return gdf

depot = create_gdf(depot_data, 'depot')
filling = create_gdf(filling_data, 'filling')

# Nigeria bounding box (same as search)
bbox = {'min_lat':4,'max_lat':14,'min_lon':2.5,'max_lon':14.5}
depot_ng = depot.cx[bbox['min_lon']:bbox['max_lon'], bbox['min_lat']:bbox['max_lat']]
filling_ng = filling.cx[bbox['min_lon']:bbox['max_lon'], bbox['min_lat']:bbox['max_lat']]

print(f"Depots in Nigeria: {len(depot_ng)}")
print(f"Filling plants in Nigeria: {len(filling_ng)}")

# Intersection based on place_id
depot_ids = set(depot_ng['place_id'])
filling_ids = set(filling_ng['place_id'])

both_ids = depot_ids & filling_ids
only_depot_ids = depot_ids - filling_ids
only_filling_ids = filling_ids - depot_ids

both = depot_ng[depot_ng['place_id'].isin(both_ids)].copy()
only_depot = depot_ng[depot_ng['place_id'].isin(only_depot_ids)].copy()
only_filling = filling_ng[filling_ng['place_id'].isin(only_filling_ids)].copy()

both['type'] = 'both'
only_depot['type'] = 'depot_only'
only_filling['type'] = 'filling_only'

# Reproject to 3857
both = both.to_crs('EPSG:3857')
only_depot = only_depot.to_crs('EPSG:3857')
only_filling = only_filling.to_crs('EPSG:3857')

# Save to existing GeoPackage (append layers)
gpkg = OUTPUT_GPKG
if not os.path.exists(gpkg):
    # Create file with first layer
    both.to_file(gpkg, layer='both', driver='GPKG')
else:
    # Append layers (if layer name exists, it will be replaced)
    both.to_file(gpkg, layer='both', driver='GPKG')
    only_depot.to_file(gpkg, layer='depot_only', driver='GPKG')
    only_filling.to_file(gpkg, layer='filling_only', driver='GPKG')

print(f"Saved: both {len(both)}, depot_only {len(only_depot)}, filling_only {len(only_filling)}")

Depots in Nigeria: 170
Filling plants in Nigeria: 603
Saved: both 103, depot_only 67, filling_only 500
